# SCLib Timeline Landscape Analysis

First-pass notebook for the 1996-2026 superconductivity scoping review.

This notebook reads the frozen API-level snapshot created by:

```bash
SCLIB_SNAPSHOT_ID=2026.05.13 node data_freeze/freeze_sclib_snapshot.mjs
```

The goal is to reproduce and formalize the central timeline observations:

- Low-Tc density below 50 K.
- The 50-80 K gap candidate.
- The 80-100 K cuprate plateau.
- Family burst events, especially iron-based superconductors in 2008.
- Theory/experiment and pressure-regime contrasts.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data_freeze').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SNAPSHOT_DIR = PROJECT_ROOT / 'data_freeze/snapshots/2026.05.13'
OUT_DIR = PROJECT_ROOT / 'analysis_outputs/2026.05.13'
OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(SNAPSHOT_DIR / 'metadata.json') as f:
    metadata = json.load(f)

timeline = pd.read_csv(SNAPSHOT_DIR / 'timeline_points.csv')
timeline_exp = pd.read_csv(SNAPSHOT_DIR / 'timeline_experimental_only_points.csv')
materials = pd.read_csv(SNAPSHOT_DIR / 'materials_default_summary.csv')
materials_all = pd.read_csv(SNAPSHOT_DIR / 'materials_all_summary.csv')

print(metadata['snapshot_id'], metadata['frozen_at'])
print(metadata['endpoint_counts'])
timeline.head()

## 1. Data Hygiene

The public timeline is already filtered by SCLib's conservative defaults. This cell applies manuscript-level temporal boundaries and derives evidence/pressure classes. Main complete-year analyses should use 1996-2025; 2026 is treated separately as partial-year data.

In [ ]:
df = timeline.copy()
df['family'] = df['family'].fillna('Other').replace('', 'Other')
df['pressure_class'] = np.select(
    [df['pressure_gpa'].isna(), df['pressure_gpa'].fillna(0) <= 1, df['pressure_gpa'].fillna(0) > 1],
    ['unknown_pressure', 'ambient_or_low_pressure', 'high_pressure'],
    default='unknown_pressure',
)
df['evidence'] = np.where(df['is_theoretical'].astype(bool), 'theoretical', 'experimental')
df['evidence_regime'] = df['evidence'] + '_' + df['pressure_class']
df['complete_year'] = df['year'].between(1996, 2025)

main = df[df['complete_year']].copy()
partial_2026 = df[df['year'] == 2026].copy()

print('main records', len(main), 'partial 2026 records', len(partial_2026))
main[['year', 'material', 'family', 'tc_kelvin', 'pressure_gpa', 'evidence_regime']].head()

## 2. Tc Bands

This reproduces the main Tc-band table and computes density by K. The planned manuscript test is whether 50-80 K is underpopulated relative to neighboring bands, both by record count and by unique material count.

In [ ]:
bands = [
    ('0-10', 0, 10), ('10-20', 10, 20), ('20-30', 20, 30),
    ('30-40', 30, 40), ('40-50', 40, 50), ('50-60', 50, 60),
    ('60-70', 60, 70), ('70-80', 70, 80), ('80-90', 80, 90),
    ('90-100', 90, 100), ('100-120', 100, 120), ('120-150', 120, 150),
    ('150-200', 150, 200), ('200-251', 200, 251),
]

rows = []
for band, lo, hi in bands:
    sub = main[(main['tc_kelvin'] >= lo) & (main['tc_kelvin'] < hi)]
    rows.append({
        'band': band,
        'lower_k': lo,
        'upper_k': hi,
        'width_k': hi - lo,
        'points': len(sub),
        'point_density_per_k': len(sub) / (hi - lo),
        'experimental_points': int((sub['evidence'] == 'experimental').sum()),
        'theoretical_points': int((sub['evidence'] == 'theoretical').sum()),
        'unique_materials': sub[['material', 'family']].drop_duplicates().shape[0],
        'unique_papers': sub['paper_id'].dropna().nunique(),
        'top_families': sub['family'].value_counts().head(6).to_dict(),
    })

tc_band_summary = pd.DataFrame(rows)
tc_band_summary.to_csv(OUT_DIR / 'tc_band_summary.csv', index=False)
tc_band_summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(tc_band_summary['band'], tc_band_summary['points'], color='#5B7DBB')
ax.set_ylabel('Timeline points')
ax.set_xlabel('Tc band (K)')
ax.set_title('SCLib timeline point density by Tc band, complete years 1996-2025')
ax.tick_params(axis='x', rotation=45)
fig.tight_layout()
fig.savefig(OUT_DIR / 'fig_tc_band_counts.png', dpi=200)
plt.show()

## 3. Gap Ratio

The first manuscript-level metric compares 50-80 K density with adjacent 30-50 K and 80-100 K regions.

In [ ]:
def band_count(frame, lo, hi):
    return frame[(frame['tc_kelvin'] >= lo) & (frame['tc_kelvin'] < hi)].shape[0]

density_30_50 = band_count(main, 30, 50) / 20
density_50_80 = band_count(main, 50, 80) / 30
density_80_100 = band_count(main, 80, 100) / 20
gap_ratio = density_50_80 / np.mean([density_30_50, density_80_100])

mat_density_30_50 = main[main['tc_kelvin'].between(30, 50, inclusive='left')][['material','family']].drop_duplicates().shape[0] / 20
mat_density_50_80 = main[main['tc_kelvin'].between(50, 80, inclusive='left')][['material','family']].drop_duplicates().shape[0] / 30
mat_density_80_100 = main[main['tc_kelvin'].between(80, 100, inclusive='left')][['material','family']].drop_duplicates().shape[0] / 20
material_gap_ratio = mat_density_50_80 / np.mean([mat_density_30_50, mat_density_80_100])

gap_metrics = pd.DataFrame([
    {'metric': 'record_gap_ratio', 'value': gap_ratio},
    {'metric': 'material_gap_ratio', 'value': material_gap_ratio},
])
gap_metrics.to_csv(OUT_DIR / 'gap_metrics.csv', index=False)
gap_metrics

## 4. Family Composition By Tc Regime

This table supports the claim that the 80-100 K plateau is cuprate-dominated, while the 0-50 K region is multi-family and the 150 K+ region is hydride/theory-heavy.

In [ ]:
regime_bins = [0, 10, 30, 50, 80, 100, 150, 251]
regime_labels = ['0-10', '10-30', '30-50', '50-80', '80-100', '100-150', '150-251']
main['tc_regime'] = pd.cut(main['tc_kelvin'], regime_bins, right=False, labels=regime_labels)
family_regime = (
    main.dropna(subset=['tc_regime'])
    .groupby(['tc_regime', 'family'], observed=True)
    .size()
    .reset_index(name='points')
)
family_regime.to_csv(OUT_DIR / 'family_tc_regime_counts.csv', index=False)

pivot = family_regime.pivot(index='tc_regime', columns='family', values='points').fillna(0)
top_cols = pivot.sum().sort_values(ascending=False).head(10).index
pivot[top_cols].plot(kind='bar', stacked=True, figsize=(11, 5), colormap='tab20')
plt.ylabel('Timeline points')
plt.xlabel('Tc regime (K)')
plt.title('Family composition by Tc regime')
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_family_tc_regime_counts.png', dpi=200)
plt.show()
pivot[top_cols]

## 5. Family Burst Detection

Burst score is defined as `(current_year_points + 1) / (previous_3_year_average + 1)`. A candidate burst requires at least 20 points, at least 3 unique materials, and burst score >= 3. Candidate bursts still need manual validation to rule out review/table artifacts.

In [ ]:
burst_source = df[df['year'].between(1993, 2025)].copy()
family_year = (
    burst_source.groupby(['family', 'year'])
    .agg(
        points=('tc_kelvin', 'size'),
        unique_materials=('material', 'nunique'),
        unique_papers=('paper_id', 'nunique'),
        experimental_points=('evidence', lambda s: int((s == 'experimental').sum())),
        theoretical_points=('evidence', lambda s: int((s == 'theoretical').sum())),
        max_tc_k=('tc_kelvin', 'max'),
    )
    .reset_index()
    .sort_values(['family', 'year'])
)

rows = []
for family, group in family_year.groupby('family'):
    lookup = dict(zip(group['year'], group['points']))
    for _, row in group.iterrows():
        prev3 = np.mean([lookup.get(row['year'] - i, 0) for i in (1, 2, 3)])
        burst_score = (row['points'] + 1) / (prev3 + 1)
        out = row.to_dict()
        out['prev3_avg_points'] = prev3
        out['burst_score'] = burst_score
        out['burst_candidate'] = bool(row['points'] >= 20 and row['unique_materials'] >= 3 and burst_score >= 3)
        rows.append(out)

family_year_bursts = pd.DataFrame(rows)
family_year_bursts = family_year_bursts[family_year_bursts['year'].between(1996, 2025)].copy()
family_year_bursts.to_csv(OUT_DIR / 'family_year_bursts.csv', index=False)
family_year_bursts[family_year_bursts['burst_candidate']].sort_values('burst_score', ascending=False).head(20)

## 6. Theory/Experiment Evidence Regimes

This section separates experimental ambient/low-pressure, experimental high-pressure, theoretical high-pressure, theoretical unknown-pressure, and unknown-pressure experimental records. Missing pressure should not be silently treated as ambient.

In [ ]:
evidence_summary = (
    main.groupby('evidence_regime')
    .agg(
        points=('tc_kelvin', 'size'),
        unique_materials=('material', 'nunique'),
        unique_papers=('paper_id', 'nunique'),
        max_tc_k=('tc_kelvin', 'max'),
    )
    .reset_index()
    .sort_values('points', ascending=False)
)
evidence_summary.to_csv(OUT_DIR / 'evidence_regime_summary.csv', index=False)
evidence_summary

In [ ]:
yearly_envelope = (
    main.groupby(['year', 'evidence_regime'])
    .agg(max_tc_k=('tc_kelvin', 'max'))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 5))
for regime, group in yearly_envelope.groupby('evidence_regime'):
    ax.plot(group['year'], group['max_tc_k'], marker='o', linewidth=1.5, label=regime)
ax.set_ylabel('Yearly maximum Tc (K)')
ax.set_xlabel('Year')
ax.set_title('Theory/experiment Tc envelope by pressure regime')
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR / 'fig_evidence_regime_tc_envelope.png', dpi=200)
plt.show()

## 7. Iron-Based 2008 Case Study

This cell prepares the monthly propagation table for the iron-based burst. The paper narrative should cite the canonical Kamihara JACS 2008 discovery and then use SCLib's arXiv timeline to show rapid propagation across LaFeAsO and rare-earth FeAsO variants.

In [ ]:
iron_2008 = main[(main['family'] == 'iron_based') & (main['year'] == 2008)].copy()
iron_2008['arxiv_month'] = iron_2008['paper_id'].str.extract(r'arxiv:(\d{4})\.')[0]
iron_2008['calendar_month'] = '20' + iron_2008['arxiv_month'].str[:2] + '-' + iron_2008['arxiv_month'].str[2:]

iron_monthly = (
    iron_2008.groupby('calendar_month')
    .agg(
        points=('tc_kelvin', 'size'),
        unique_materials=('material', 'nunique'),
        unique_papers=('paper_id', 'nunique'),
        max_tc_k=('tc_kelvin', 'max'),
    )
    .reset_index()
)
iron_monthly.to_csv(OUT_DIR / 'iron_based_2008_monthly.csv', index=False)
iron_monthly

## 8. Next Validation Outputs

Before drafting final claims, generate stratified validation samples from this notebook or from a database-level export. Minimum validation strata are defined in `review_protocol/scope_review_protocol.md`.